In [13]:
%load_ext autoreload
%autoreload 2

# Import the necessary libraries 
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

from src_prognostics.utils import load_data, format_data, save_prognostics
from src_prognostics.hsmm import CustomHSMM, predict_hsmm_pdf_staked, predict_hsmm_bounds
from src_prognostics.metrics import mae, picp, pinaw

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Assignment

You recive a dataset that contains condition monitoring (CM) data from a structure. The dataset contrains several run to failure experiments that will allow you to train a hidden semi-Markov model (HSMM) for remaining useful life (RUL) prognostics. In addition to the CM data, dataset indicates the operational conditions in which the expreminets were perfomed.

This workshop considers the following tasks:
1. Visulize the provided data, explore its contents and gather some insights
2. Use the provided codes to train a baseline HSMM. Reflect on the achieved accuracy and uncertainty representation.
3. Reflect on how you incorporate uncertainty management in the HSMM.
4. Implemet your uncertainty management strategy and analyze its effectivity
5. Compute all prognostic distributions for all training, validation and testing datasets. These results will be latter used for deicission-making.

The following cells contain some code examples that you could use. Also, some functionalies are already implemeted.in the source code. Feel free to use them. 
In case you have questions, or need assistence, don't hessitate to ask for help 😁


## Task 1: Analyze the data

In [14]:
# To load the dataset
df_train = load_data('train')
df_train.head()

df_validation = load_data('validation')
df_validation.head()

df_test = load_data('test')
df_test.head()

,hi,load,exp_id
0,-0.000000,0.0,0.0
1,-0.001783,0.0,0.0
2,-0.001677,0.0,0.0
3,-0.001381,0.0,0.0
4,0.000813,0.0,0.0


In [ ]:
# Plot some columns of the dataset here

In [ ]:
# Plot some columns of the dataset here

In [ ]:
# Plot some columns of the dataset here

What insigts did you get from the data?

Write your answer here:



...

## Task 2: Train an HSMM

Here are some implemented codes that can train an HSMM

In [15]:
# Transform the data to the HIMAP format
data_train, max_len = format_data(df_train)

In [16]:
# Instance the model with some hiper-parameters
n_states = 4
model = CustomHSMM(n_states=n_states, # You can change this
                   n_durations = (max_len//n_states)*2, # You can change this
                   f_value=-0.34, # Please, dont change this.
                   obs_state_len=len(data_train.keys()), # Please, dont change this.
                   n_iter=3, # You can change this
                   name = 'hsmm_exmple' # You can change this
                   )

Created folder: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results
Created folder: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\dictionaries
Created folder: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\figures
Created folder: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models


In [17]:
# Fit the model parameters
model.fit(data_train)

Iters 1/3: 100%|██████████| 40/40 [00:10<00:00,  3.94it/s]



FIT hsmm_exmple: re-estimation complete for loop 1 with score: -5687.239309463148.


Iters 2/3: 100%|██████████| 40/40 [00:09<00:00,  4.07it/s]



FIT hsmm_exmple: re-estimation complete for loop 2 with score: -215.5170455001782.


Iters 3/3: 100%|██████████| 40/40 [00:09<00:00,  4.26it/s]


FIT hsmm_exmple: re-estimation complete for loop 3 with score: -220.59265433409482.
[0, 1, 2, 3]


In [19]:
# Save the model for later
model.save_model()

Model saved to c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models\hsmm_exmple.pkl.


In [20]:
# In case you need to load a trained model
model = CustomHSMM(name = 'hsmm_exmple')
model.load_model('hsmm_exmple')

Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\dictionaries
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\figures
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models


In [21]:
alpha = 0.005 # Confidence level used to generate the intervals
print('Mae\tPINAW\tPICP')
for cmd in data_train.values():
    
    actual_RUL = len(cmd)-np.arange(len(cmd))
    
    expected, lb, ub = predict_hsmm_bounds(cmd, model,alpha = alpha ) # Function that returns the expected rul and lower and upper values for a confidence intervals
    
    print(mae(expected,actual_RUL), '\t', pinaw(lb,ub),'\t', picp(lb,ub, actual_RUL))
    
    # # Uncomment these lines to get the RUL plot
    # x_axis = np.arange(len(cmd))
    # plt.figure()
    # plt.plot(x_axis,expected, label = 'Prognostic', color = 'b')
    # plt.fill_between(x_axis, lb, ub, alpha = 0.2, color = 'b')
    # plt.plot(x_axis,actual_RUL, color = 'r', label = 'Actual RUL')
    # plt.ylabel('RUL')
    # plt.xlabel('Timestep')
    # plt.legend()
    # plt.show()
    # break

    
    

Mae	PINAW	PICP
22.763052350317697 	 111.63057324840764 	 0.7452229299363057
28.49629445983235 	 112.57738095238095 	 0.7619047619047619
33.97538374517747 	 118.17297297297297 	 0.7837837837837838
33.79978363962419 	 112.05084745762711 	 0.7740112994350282
28.460020767254402 	 114.00588235294117 	 0.7647058823529411
32.442036818014806 	 115.66480446927375 	 0.776536312849162
36.970306381651774 	 113.53513513513514 	 0.7837837837837838
30.12265238391464 	 117.44067796610169 	 0.7740112994350282
30.755954240847277 	 113.34682080924856 	 0.7687861271676301
26.803609192505952 	 112.58181818181818 	 0.7575757575757576
34.31583277442666 	 118.39784946236558 	 0.7849462365591398
33.80484606474065 	 114.47777777777777 	 0.7777777777777778
30.42404923194816 	 108.73053892215569 	 0.7604790419161677
34.47167330444096 	 119.02139037433155 	 0.786096256684492
29.000173061572703 	 113.33529411764705 	 0.7647058823529411
32.6485397270071 	 118.43169398907104 	 0.7814207650273224
32.199671770919295 	 

Comment on the uncertainty of the prognostics.

Write your answer here:



...


## Task 3: Reflect on how could you manage prognostic uncertianty

Write your answer here:



...

## Task 4: Implement your uncertainty management strategy


Hint: use the codes from Task 2

## Task 5: Compute your prognostics for the training, validation and testing datasets

In [ ]:
# Example on how to generate prognostics with a single model

# 1. Load the model
model = CustomHSMM(name = 'hsmm_exmple')
model.load_model('hsmm_exmple')

# 2. format the dataset
data_train, _ = format_data(df_train)

# 3.  Genrate the prognostics for each trajectory
predictions_train = []
for trajectory in data_train.values():
    predictions_train = predict_hsmm_pdf_staked(trajectory, model,predictions_train)
    
# 4. Save the prognostics
save_prognostics(predictions_train, 'train')


Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\dictionaries
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\figures
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models
train saved succesfully!


In [24]:
# For validation
# 2. format the dataset
data_val, max_len = format_data(df_validation)

# 3.  Genrate the prognostics for each trajectory
predictions_val = []
for trajectory in data_val.values():
    predictions_val = predict_hsmm_pdf_staked(trajectory, model,predictions_val)
    
# 4. Save the prognostics
save_prognostics(predictions_val, 'validation')

validation saved succesfully!


In [25]:
# For test
# 2. format the dataset
data_test, max_len = format_data(df_test)

# 3.  Genrate the prognostics for each trajectory
predictions_test = []
for trajectory in data_test.values():
    predictions_test = predict_hsmm_pdf_staked(trajectory, model,predictions_test)
    
# 4. Save the prognostics
save_prognostics(predictions_test, 'test')

test saved succesfully!
